# Lab: Tune Hyperparameters
## Purpose:
- Implement neural networks in Keras.
- Understand how to interpret loss curves for the training process.

### Topics:
- Keras
- Plotting training and test loss curves
- Decision boundaries
-

### Steps
* Define an MLP in Keras.
* Train the MLP on the embeddings dataset.
* Plot the training and test loss curve.
* Visualize the decision boundary of the model.

Date: 2026-02-21

Source: https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_3/gdm_lab_3_5_tune_hyperparameters.ipynb

References: https://github.com/google-deepmind/ai-foundations
- GDM GH repo used in AI training courses at the university & college level.

In [ ]:
%%capture
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os # For adjusting Keras settings.
os.environ['KERAS_BACKEND'] = 'jax' # Set a parameter for Keras.

# Packages used.
import jax.numpy as jnp # For defining matrices.
import keras # For defining your MLP.
import pandas as pd # For loading the dataset.
# For splitting data into train and test splits.
from sklearn import model_selection

from ai_foundations import machine_learning # For training your MLP.
from ai_foundations import visualizations # For visualizing data and boundaries.
from ai_foundations import training # For logging the loss during training.
from ai_foundations.feedback import course_3 # For providing feedback.

# Generating the data

Load the embeddings and labels.

The below line automatically splits the data into training and testing sets.
- Remember that the original dataset was split into [:, -1] (the prompt) and [1, :] (the target).

```python
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42
)
```

Randomly splits the dataset such that 80% of the examples are in the training set and 20% are in the test set.

In [ ]:
# Load data using pandas.
df = pd.read_csv("https://storage.googleapis.com/dm-educational/assets/ai_foundations/mat-apple-bank-dataset.csv")

# Extract embeddings (Embedding_dim_1, Embedding_dim_2) and labels.
X = jnp.array(df[["Embedding_dim_1", "Embedding_dim_2"]].values)
# Labels: 0 ("mat"), 1 ("apple"), or 2 ("bank").
y = jnp.array(df["Label"].values)

# Human-readable labels.
labels = ["mat", "apple", "bank"]

# Randomly split dataset into training and test sets.
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert data into JAX arrays so that they can be used to train your MLP model.
X_train, y_train = jnp.array(X_train), jnp.array(y_train)
X_test, y_test = jnp.array(X_test), jnp.array(y_test)

## Optimizing hyperparameters

-----
>Experiment with networks that differ in terms of:
> * the number of layers
> * the dimensions of each layer (how many neurons there are in each layer)
>
>The model is likely to overfit if
> 1. there are many layers
> 2. The hidden layer dimensions are large
>
>Too few layers + too low dimensions = underfit
>
>- Keep everything but the number of layers and the dimensions of the layers constant.
>Don't change the number of epochs. This allows you to directly attribute any differences in model performance to the hyperparameters that you change.
>
>The cells below define different network structures by changing `hidden_dims`.
>
>1. Visualize the structure of the network to get a sense of what it looks like.
>2. Train the network on the dataset.
>A ccuracy is defined as the percentage of examples for which the model makes the correct predictions.
>Train accuracy indicates whether the model learned something — low train accuracy suggests **underfitting**.
>The test accuracy indicates whether the model is able to generalize — low test accuracy in combination with high train accuracy suggests **overfitting**.
>
>
>Repeat these three steps with at least three network configurations (three different settings of `hidden_dims`).
>
>Make sure to train:
>1. one network with only one layer and a very low number of neurons
>2. one network that has at least three layers and the first layer has more than 1,000 neurons
>3. one network with settings somewhere in between
>
-----

In [ ]:
# Set the number of hidden layers and their dimensions here.
# Each number indicates the number of neurons per layer. For example, setting
# this to `[20, 10]` will initialze an MLP with two hidden layers of dimensions
# 20 and 10, respectively.
hidden_dims = [20, 10]

# Initialize the model and visualize its structure.
mlp_model = machine_learning.build_mlp(hidden_dims=hidden_dims, n_classes=3)
visualizations.visualize_mlp_architecture(hidden_dims, 3)

# Train the model.
training_logger = training.CustomAccuracyPrinter(print_every=20)
training_history = machine_learning.train_mlp(
    mlp_model,
    X_train,
    y_train,
    learning_rate=0.005,
    validation_data=(X_test, y_test),
    epochs=500,
    callbacks=[training_logger],
)

# Plot the loss curve.
visualizations.plot_loss_curve(training_history.history)

# Compute the accuracy and visualize the decision boundaries.
train_acc = training_history.history["accuracy"][-1]
test_acc = training_history.history["val_accuracy"][-1]

print(f"Train accuracy: {train_acc * 100:.2f}%")
print(f"Test accuracy: {test_acc * 100:.2f}%")

visualizations.plot_data_and_mlp(
    X_train,
    y_train,
    labels,
    features_test=X_test,
    label_ids_test=y_test,
    mlp_model=mlp_model,
    title="Decision Boundaries - Your MLP",
)